# Кривая чувствительности без лампы — по шагам

**Идея.** Если две линии идут с одного верхнего уровня *k*, то число фотонов в них относится как
их коэффициенты Эйнштейна: N₁/N₂ = A₁/A₂. Это не зависит ни от ne, ни от Te, ни от того, как
заселён уровень. Спектрометр же регистрирует не N, а N·S(λ), где S — чувствительность. Поэтому

    площадь линии = S(λ) · n_k · A · (константа)   →   ln(площадь / A) = ln S(λ) + [своё число для уровня k]

Для каждой линии мы знаем площадь (меряем) и A (NIST). Неизвестны: гладкая функция ln S(λ) и
по одному числу на каждый верхний уровень (его заселённость). Всё это находится одним
линейным МНК.

Шаги:
1. загрузить спектр и таблицы NIST;
2. выбрать кандидатов (у линии есть A, она не резонансная, рядом нет других линий);
3. подогнать гауссиану к каждой — площадь, ширина, S/N; выкинуть плохие пики;
4. оставить только уровни, у которых ≥ 2 хороших линии;
5. МНК: ln(площадь/A) = c₁x + c₂x² + c₃x³ + b_уровня;
6. выкинуть выбросы, перегнать, нарисовать S(λ).

Каждый шаг — отдельная ячейка: меняйте пороги и смотрите, что меняется.
Это та же логика, что в `sensitivity_no_lamp.py`, только развёрнутая.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings("ignore")

SPECTRUM = "A3.xlsx"      # ваш файл со спектром
SHEET = "data10"          # лист: выдержка, в которой нужные линии не насыщены
NIST_DIR = "../../docs/crm"   # папка с nist_*.csv (выгрузки lines1.pl)

FWHM = 0.0245             # аппаратная ширина, нм (мерили по одиночным линиям)
SAT = 64000               # уровень насыщения детектора, отсчёты
MIN_SN = 30               # минимальное сигнал/шум
BAD_ACC = {"D+", "D", "E"}  # классы точности A в NIST, которые не берём (ошибка > 40-50 %)
WMIN, WMAX = 400, 816     # диапазон спектра, нм

## 1. Спектр и NIST

In [ ]:
sp = pd.read_excel(SPECTRUM, sheet_name=SHEET)
w = sp.iloc[:, 0].to_numpy(float)   # длина волны, нм
I = sp.iloc[:, 1].to_numpy(float)   # отсчёты
plt.figure(figsize=(12, 3)); plt.plot(w, I, lw=0.5); plt.xlabel("λ, нм"); plt.ylabel("отсчёты"); plt.show()

In [ ]:
def clean(s):
    # NIST пишет ячейки как ="123.45" — снимаем кавычки и знак =
    return str(s).replace('"', '').lstrip('=').strip() if pd.notna(s) else ""

def num(s):
    s = re.sub(r"[^0-9.eE+-]", "", clean(s))
    try:
        return float(s)
    except ValueError:
        return np.nan

def read_nist(species):
    d = pd.read_csv(f"{NIST_DIR}/nist_{species}.csv", dtype=str)
    out = pd.DataFrame({
        "sp": species,
        "wl": [num(a) if not np.isnan(num(a)) else num(b) for a, b in zip(d["obs_wl_air(nm)"], d["ritz_wl_air(nm)"])],
        "I_nist": [int(m.group()) if (m := re.match(r"\d+", clean(x))) else 0 for x in d["intens"]],
        "A": [num(x) for x in d["Aki(s^-1)"]],
        "acc": [clean(x) for x in d["Acc"]],
        "Ei": [num(x) for x in d["Ei(cm-1)"]],
        "Ek": [num(x) for x in d["Ek(cm-1)"]],
    })
    return out.dropna(subset=["wl"])

# все элементы, которые могут дать бленду, и те, чьи линии мы используем
ALL = ["XeI", "XeII", "XeIII", "BaI", "BaII", "WI", "WII", "CaI", "CaII", "AlI", "AlII"]
USE = ["XeI", "XeII", "BaI", "BaII"]
nist = pd.concat([read_nist(s) for s in ALL], ignore_index=True)
nist.groupby("sp").size()

## 2. Кандидаты (только по таблице NIST, спектр ещё не смотрим)

Правила и зачем они нужны:

| правило | почему |
|---|---|
| у линии есть A в NIST | без A не с чем сравнивать |
| класс точности не D+/D/E | ошибка A прямо идёт в S; у D/E это 50 % и больше |
| нижний уровень ≠ основной (Ei ≠ 0) | резонансную линию плазма может сама поглощать — N₁/N₂ ≠ A₁/A₂ |
| в ±1.5 FWHM нет другой линии NIST (заметной или с A) | иначе в площадь попадёт чужая линия |

In [ ]:
cand = nist[nist.sp.isin(USE) & nist.A.notna() & nist.Ek.notna() & nist.wl.between(WMIN, WMAX)].copy()
cand["reason"] = ""

def mark(mask, why):
    # пишем причину только тем, кого ещё не отбросили
    m = mask & (cand.reason == "")
    cand.loc[m, "reason"] = why

mark(cand.acc.isin(BAD_ACC), "плохой класс точности A")
mark(cand.Ei == 0, "резонансная")

def neighbour(row):
    near = nist[(abs(nist.wl - row.wl) <= 1.5 * FWHM)]
    near = near[~((near.sp == row.sp) & (abs(near.wl - row.wl) < 1e-4))]          # сама линия
    near = near[(near.I_nist >= 0.1 * max(1, row.I_nist)) | near.A.notna()]          # заметные соседи
    return "" if near.empty else f"бленда с {near.iloc[0].sp} {near.iloc[0].wl:.3f}"

for i, row in cand[cand.reason == ""].iterrows():
    b = neighbour(row)
    if b:
        cand.loc[i, "reason"] = b
print(len(cand), "линий с A в диапазоне;", (cand.reason == "").sum(), "прошли шаг 2")

## 3. Подгонка каждой линии гауссианой

Около табличной λ ищем самый высокий отсчёт (шкала спектрометра сдвинута, поэтому окно
[λ − 0.13; λ + 0.02] нм), берём ±4 точки, фон — 25-й перцентиль в ±1.5 нм, подгоняем
`a·exp(−(x−c)²/2s²) + фон`. Площадь ∝ a·FWHM (для гауссианы площадь = 1.064·a·FWHM;
множитель общий для всех линий и уходит в нормировку).

Шум — разброс соседних точек (робастно, через медиану) в ±1 нм.

Отбрасываем: насыщенные (пик ≥ SAT); ширина не 0.6–1.6 × FWHM (значит бленда или не линия);
S/N < MIN_SN.

In [ ]:
def gauss_bg(x, a, c, s, bg):
    return a * np.exp(-0.5 * ((x - c) / s) ** 2) + bg

def fit_line(wl):
    m = (w > wl - 0.13) & (w < wl + 0.02)
    if not m.any():
        return None
    k = np.flatnonzero(m)[np.argmax(I[m])]               # индекс максимума
    x, y = w[k - 4:k + 5], I[k - 4:k + 5]
    bg = np.percentile(I[(w > w[k] - 1.5) & (w < w[k] + 1.5)], 25)
    try:
        p, _ = curve_fit(lambda x, a, c, s: gauss_bg(x, a, c, s, bg), x, y, p0=[I[k] - bg, w[k], FWHM / 2.3548])
        amp, fwhm = p[0], 2.3548 * abs(p[2])
    except RuntimeError:
        amp, fwhm = I[k] - bg, np.nan
    d = np.diff(I[(w > w[k] - 1) & (w < w[k] + 1)])
    noise = 1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2)
    return dict(peak=I[k], amp=amp, fwhm=fwhm, sn=amp / noise)

for i, row in cand[cand.reason == ""].iterrows():
    f = fit_line(row.wl)
    if f is None:
        cand.loc[i, "reason"] = "нет пика"; continue
    for key, v in f.items():
        cand.loc[i, key] = v
    if f["peak"] >= SAT:
        cand.loc[i, "reason"] = "насыщена"
    elif not (0.6 < f["fwhm"] / FWHM < 1.6):
        cand.loc[i, "reason"] = f"ширина {f['fwhm']*1e3:.0f} пм"
    elif f["sn"] < MIN_SN or f["amp"] <= 0:
        cand.loc[i, "reason"] = f"слабая, S/N={f['sn']:.0f}"
cand["area"] = cand.amp * cand.fwhm
print((cand.reason == "").sum(), "линий прошли шаг 3")

In [ ]:
# посмотреть любую линию глазами
def show(wl, half=0.3):
    m = (w > wl - half) & (w < wl + half)
    plt.plot(w[m], I[m], "o-", ms=3); plt.axvline(wl, color="r", lw=0.5)
    plt.title(f"{wl} нм"); plt.show()
show(541.915)

## 4. Группы по верхнему уровню

Верхний уровень определяем по Ek (и элементу). Уровень с одной хорошей линией не даёт
никакой информации о S (у него своё неизвестное b), поэтому выкидываем.

In [ ]:
good = cand[cand.reason == ""].copy()
good["level"] = good.sp + " " + good.Ek.round(1).astype(str)
n = good.groupby("level").wl.transform("size")
cand.loc[good.index[n < 2], "reason"] = "одна линия на уровне"
good = good[n >= 2]
good["y"] = np.log(good.area / good.A)          # вот что подгоняем
good.sort_values(["level", "wl"])[["sp", "wl", "level", "A", "acc", "area", "sn", "y"]]

### Пример руками: уровень Xe II 113512.4 см⁻¹ (это уровень 29 у Ванга)

Три линии: 541.915, 553.107, 571.961 нм. Отношение S двух линий:

    S(λ₂)/S(λ₁) = (площадь₂/A₂) / (площадь₁/A₁)

553.107 в группу не попадёт: в 1.5 FWHM от неё есть W I 553.138 (шаг 2 проверяет соседей по
всем элементам из `ALL`). Причину любой линии можно посмотреть так:
`cand[abs(cand.wl - 553.107) < 0.01]`.

In [ ]:
g = good[good.level.str.startswith("XeII 113512")]
ref = g.iloc[0]
for _, r in g.iterrows():
    print(f"{r.wl:8.3f}  площадь={r.area:8.1f}  A={r.A:.2e}  S/S({ref.wl:.1f}) = {(r.area/r.A)/(ref.area/ref.A):.3f}")

## 5. МНК

Неизвестные: c₁, c₂, c₃ (форма ln S) и b₁…b_L (по одному на уровень).
x = (λ − 550)/150 — чтобы числа были порядка 1. Свободного члена у многочлена нет:
он неотличим от b, поэтому S(550) = 1 по определению — кривая относительная.

Матрица: строка = линия; столбцы = [x, x², x³, 0…1…0] (единица в столбце своего уровня).

In [ ]:
ORDER = 3

def solve(df):
    levels = sorted(df.level.unique())
    x = (df.wl.to_numpy() - 550) / 150
    M = np.column_stack([x ** k for k in range(1, ORDER + 1)] +
                        [(df.level == L).to_numpy(float) for L in levels])
    c, *_ = np.linalg.lstsq(M, df.y.to_numpy(), rcond=None)
    res = df.y.to_numpy() - M @ c
    dof = max(len(df) - len(c), 1)
    cov = (res @ res / dof) * np.linalg.pinv(M.T @ M)
    return c, res, cov

c, res, cov = solve(good)
good["res"] = res
print("разброс ln S:", np.std(res, ddof=0).round(3))

## 6. Выбросы

Робастная σ = 1.4826 · медиана|остатков| (обычное std раздувается самими выбросами).
Точки дальше 3σ убираем один раз, уровни с одной оставшейся линией — тоже, и перегоняем.

In [ ]:
sig = 1.4826 * np.median(np.abs(good.res))
out = good[np.abs(good.res) > 3 * sig]
print("выбросы:", list(zip(out.sp, out.wl.round(3), out.res.round(2))))
cand.loc[out.index, "reason"] = "выброс"
good = good.drop(out.index)
good = good[good.groupby("level").wl.transform("size") >= 2]
c, res, cov = solve(good)
good["res"] = res
s = np.sqrt(res @ res / max(len(good) - len(c), 1))
print(f"{len(good)} линий, {good.level.nunique()} уровней, разброс ≈ {100*(np.exp(s)-1):.0f} %")

In [ ]:
lam = np.arange(good.wl.min(), good.wl.max() + 1, 5.0)
B = np.column_stack([((lam - 550) / 150) ** k for k in range(1, ORDER + 1)])
lnS = B @ c[:ORDER]
err = np.sqrt(np.einsum("ij,jk,ik->i", B, cov[:ORDER, :ORDER], B))

# точки: каждую линию сдвигаем на b её уровня — тогда все ложатся на одну кривую
b = dict(zip(sorted(good.level.unique()), c[ORDER:]))
good["S_point"] = np.exp(good.y - good.level.map(b))

plt.figure(figsize=(9, 5))
plt.fill_between(lam, np.exp(lnS - err), np.exp(lnS + err), alpha=0.3)
plt.plot(lam, np.exp(lnS), "k")
for L, gg in good.groupby("level"):
    plt.plot(gg.wl, gg.S_point, "o-", label=L)
plt.xlabel("λ, нм"); plt.ylabel("S / S(550)"); plt.legend(fontsize=7); plt.grid(alpha=0.3); plt.show()
pd.DataFrame({"wl": lam, "S": np.exp(lnS), "S_low": np.exp(lnS - err), "S_high": np.exp(lnS + err)}).iloc[::10]

## Итоги и что можно попробовать

* `cand.reason.value_counts()` — почему отброшены линии; `cand[cand.reason == ""]` — что взято.
* Сравните с `docs/crm/sensitivity_A3_points.csv` / `_curve.csv` (результат скрипта).
* Поменяйте `SHEET` на другой лист — кривая не должна меняться (проверка, что нет насыщения
  и нелинейности).
* `ORDER = 2` или `4` — насколько кривая зависит от формы многочлена.
* `BAD_ACC = {"C", "D+", "D", "E"}` — только точные A; точек меньше, но они надёжнее.
* W, Ca, Al в наших спектрах не видны (проверяли по их сильным линиям). Если убрать их из `ALL`,
  проверка на бленды станет мягче и вернёт часть линий (например, Xe II 553.107). Решать вам:
  строже — меньше точек, но меньше риск.
* Точность кривой не лучше точности A из NIST (класс B ≈ 10 %, C ≈ 25 %).

In [ ]:
cand.reason.replace("", "взята").str.replace(r"[\d.=,]+.*", "", regex=True).str.strip().value_counts()